# R1: the g81 body into the Hugging Face store, the lost cells rebuilt, the rows audited

`docs/RESUME-G81.md`, step R1. **CPU runtime is enough; no GPU, no model.** About 1 to 2 hours, most of it reading
3.4 GB from Drive and writing it to the store. It runs `scripts/r1_run.py`; every step can be run again safely.

Before you start:
1. In Colab, open **Secrets** (the key icon on the left) and add `HF_TOKEN`: a Hugging Face **write** token of the
   account that will own the store. Turn on notebook access for it. The token is never printed.
2. Run the cells in order. **Cell 5 stops before anything is uploaded**: read its result before cell 6.

What comes out, in a new folder `R1-results-<time>` inside the Drive recovery folder: `r1-report.json`,
`QUARANTINE.jsonl`, `web-rows.tsv`, `validation.json`. Nothing in the recovery folder is changed or deleted.

In [ ]:
# Settings
HF_REPO = "asrinna118/hivemind-cells"   # the store (a private model repository; made private if it does not exist)
R1_REF  = "main"                         # the commit of HiveMIND0.2 to run (a branch or a full SHA)

In [ ]:
# 1. Drive, and the recovery folder
from google.colab import drive
drive.mount("/content/drive")
import glob, os, time
found = glob.glob("/content/drive/MyDrive/HiveMIND recovery*2026-11-26")
assert len(found) == 1, f"the recovery folder was not found exactly once: {found}"
DRIVE = found[0]
assert os.path.exists(os.path.join(DRIVE, "RECOVERY-MANIFEST.json")) and os.path.exists(os.path.join(DRIVE, "RESTORE.py"))
OUT = os.path.join(DRIVE, "R1-results-" + time.strftime("%Y%m%d-%H%M%S", time.gmtime()))
print("recovery folder:", DRIVE)
print("results will go to:", OUT)

In [ ]:
# 2. The code (public repository, no token), and only what R1 needs (no torch: R1 runs no model)
import subprocess
subprocess.run(["rm", "-rf", "/content/HiveMIND0.2"], check=True)
subprocess.run(["git", "clone", "-q", "https://github.com/Arnav123-s/HiveMIND0.2", "/content/HiveMIND0.2"], check=True)
subprocess.run(["git", "-C", "/content/HiveMIND0.2", "checkout", "-q", R1_REF], check=True)
SHA = subprocess.run(["git", "-C", "/content/HiveMIND0.2", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
assert os.path.exists("/content/HiveMIND0.2/scripts/r1_run.py"), "this commit has no scripts/r1_run.py"
subprocess.run(["pip", "install", "-q", "numpy", "safetensors==0.8.0", "huggingface_hub==2.4.0", "zstandard==0.25.0", "beautifulsoup4==4.15.0"], check=True)
print("HiveMIND0.2 at", SHA)

In [ ]:
# 3. The store: token from Colab's secrets (never printed), the repository private
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["HIVE_STORE"] = "hf"
os.environ["HIVE_HF_REPO"] = HF_REPO
from huggingface_hub import HfApi
api = HfApi(token=os.environ["HF_TOKEN"])
print("token belongs to:", api.whoami()["name"])
from huggingface_hub.errors import RepositoryNotFoundError
try:
    info = api.repo_info(HF_REPO, repo_type="model")
except RepositoryNotFoundError:
    info = None
    print(HF_REPO, "does not exist yet: the first upload makes it, private")
if info is not None:
    assert info.private, f"{HF_REPO} exists and is PUBLIC: stop; make it private first (weights stay private)"
    print(HF_REPO, "exists, private")

In [ ]:
# 4. The driver's own tests, on this machine (about a minute; the converter used by the tests needs transformers)
subprocess.run(["pip", "install", "-q", "torch", "--index-url", "https://download.pytorch.org/whl/cpu"], check=False)
subprocess.run(["pip", "install", "-q", "transformers==5.22.2", "pytest"], check=True)
r = subprocess.run(["python", "-m", "pytest", "-q", "-p", "no:cacheprovider", "tests/test_r1_run.py"], cwd="/content/HiveMIND0.2")
assert r.returncode == 0, "the R1 tests failed on this machine: stop and report"

In [ ]:
# 5. Restore, extract, VALIDATE (the gate), audit. Nothing is uploaded yet.
r = subprocess.run(["python", "scripts/r1_run.py", "--drive", DRIVE, "--work", "/content/r1", "--out", OUT,
                    "--steps", "restore,extract,validate,audit,report", "--no-upload"], cwd="/content/HiveMIND0.2")
assert r.returncode == 0, "R1 stopped: read the lines above (a hash that does not match stops it on purpose)"
import json
st = json.load(open("/content/r1/r1-state.json"))
print("\nVALIDATION GATE:", st["validate"]["gate"], "| g82 trunk:", st["validate"]["g82_trunk"])
print("incident lines 1068-1074 flagged:", st["audit"]["incident_lines_1068_1074_flagged"], "| confirmed:", st["audit"]["incident_confirmed"])
print("incidence by kind:", json.dumps(st["audit"]["incidence_by_kind"], indent=1))
# STOP HERE and read the gate. PASS: the two rebuilt .safetensors are staged. FAIL: they are not, and the trunk
# falls back to 16 leaves (manager ruling). Either way the 84 verified files are uploaded in the next cell.

In [ ]:
# 6. Upload into the store (one commit per generation, each read back and compared), then the final report
r = subprocess.run(["python", "scripts/r1_run.py", "--drive", DRIVE, "--work", "/content/r1", "--out", OUT,
                    "--steps", "upload,report"], cwd="/content/HiveMIND0.2")
assert r.returncode == 0, "the upload stopped: read the lines above"
print(open(os.path.join(OUT, "r1-report.json")).read()[:4000])
print("\nresults in Drive:", OUT, sorted(os.listdir(OUT)))